# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishigupgta1234-ux/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Finding 1: Declining-page label

The paper uses a decline label based on the page's observed trend direction.

My methodology question is: how is this label created, and does it represent the actual content-refresh opportunity we want to identify? I would check whether the label is a useful proxy for pages that need review and clearly state its limitations.

### Finding 2: Model performance

The paper reports that the learned model performs better than the hand-written baseline when ranking pages for review.

My methodology question is: does the validation design support this comparison? I would check whether pages from the same client can appear in both training and test data. A client-level holdout would make the comparison more conservative and help show whether the result also holds for clients not seen during training.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

For the comparison, I first use a random row-level split as the weaker validation setup. This can place pages from the same client in both training and test data.

I then use a client-level holdout, where all pages from a client stay in either training or test data. This is a more conservative validation design for my question because the model is evaluated on clients it has not seen during training.

I compare both using Precision@50 with the same Decision Tree model and the same feature set.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Before: random row-level split

import pandas as pd

url = "https://raw.githubusercontent.com/ishigupgta1234-ux/flyrank-ml-internship/refs/heads/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df["label"] = (df["trend_direction"] == "down").astype(int)

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

print("Rows:", len(df))
print("Features used:", len(features))

Rows: 30000
Features used: 17


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

train_before, test_before = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

model_before = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model_before.fit(
    train_before[features].fillna(0),
    train_before["label"]
)

test_before = test_before.copy()

test_before["score"] = model_before.predict_proba(
    test_before[features].fillna(0)
)[:, 1]

top50_before = test_before.sort_values(
    "score",
    ascending=False
).head(50)

precision_before = top50_before["label"].mean()

print("Before - Random row split")
print("Precision@50:", precision_before)

Before - Random row split
Precision@50: 0.92


In [10]:
clients = df["client_id"].unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_after = df[df["client_id"].isin(train_clients)].copy()
test_after = df[df["client_id"].isin(test_clients)].copy()

model_after = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model_after.fit(
    train_after[features].fillna(0),
    train_after["label"]
)

test_after["score"] = model_after.predict_proba(
    test_after[features].fillna(0)
)[:, 1]

top50_after = test_after.sort_values(
    "score",
    ascending=False
).head(50)

precision_after = top50_after["label"].mean()

print("After - Client-level split")
print("Precision@50:", precision_after)

print(
    "Client overlap:",
    len(set(train_after["client_id"]) &
        set(test_after["client_id"]))
)

After - Client-level split
Precision@50: 0.68
Client overlap: 0


In [11]:
comparison = pd.DataFrame({
    "Validation": [
        "Random row split",
        "Client-level split"
    ],
    "Precision@50": [
        precision_before,
        precision_after
    ]
})

print(comparison)

           Validation  Precision@50
0    Random row split          0.92
1  Client-level split          0.68


The random row-level split gives a less conservative estimate because pages from the same client can appear in both training and test data. The client-level split removes this overlap and gives a more realistic estimate of performance on unseen clients.

I will use the client-level Precision@50 as the more trustworthy result for my model. This is a validation result for decision-support, not proof that the model will improve content performance in production.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



I checked the final feature set for signals that could reveal the target directly or use information that would only be available after the outcome.

The target is created from `trend_direction == "down"`, so `trend_direction` and `trend_pct` are not used as model features.

The final feature set contains page-level signals such as traffic, visibility, content age, freshness and engagement. These are available as input signals and are not directly derived from the target.

No future-window features or label-derived features are included in the final feature set.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage audit

print("Final features used by the model:")
print()

for feature in features:
    print(feature)

print()
print("Target column: trend_direction")
print()

leakage_columns = [
    "trend_direction",
    "trend_pct"
]

print("Columns excluded because they are related to the target:")
for column in leakage_columns:
    print(column)

print()
print("Leakage check:")

for column in leakage_columns:
    if column in features:
        print(column, "FOUND - possible leakage")
    else:
        print(column, "NOT USED")

Final features used by the model:

search_volume
competition
cpc
word_count
char_count
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
engaged_sessions_90d
content_age_days
days_since_last_update
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct

Target column: trend_direction

Columns excluded because they are related to the target:
trend_direction
trend_pct

Leakage check:
trend_direction NOT USED
trend_pct NOT USED


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

The Decision Tree performs better than my Week-4 baseline at identifying pages that are declining.

### Safer claim

In my validation data, the Decision Tree showed higher Precision@50 than the Week-4 baseline under the tested split. This is an observed and measured result on the available data. The result is directional and can support content-review prioritization, but it does not prove that the model will improve content performance in production.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.